In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn.functional as F
import pydicom
from transformers import AutoTokenizer, AutoModel
from skimage.transform import resize

In [ ]:
# ==============================
# PATHS (UPDATE IF NEEDED)
# ==============================
BASE_PATH = "/content/drive/My Drive/MediqaRadar02"
IMAGE_PATH = os.path.join(BASE_PATH, "image_dev")
REPORT_PATH = os.path.join(BASE_PATH, "dev_preliminary_report")
EDITS_JSON = os.path.join(BASE_PATH, "dev_edits.json")

In [ ]:
# ==============================
# LOAD EDITS
# ==============================
with open(EDITS_JSON, "r") as f:
    edits_data = json.load(f)

print(f"Loaded {len(edits_data)} cases")


Loaded 20 cases


In [ ]:
GT_PATH = "/content/drive/My Drive/RADAR-main/RADAR-main/eval/groundtruth_dev.csv"

In [ ]:
import pandas as pd
with open(EDITS_JSON, "r") as f:
    edits_data = json.load(f)

gt_df = pd.read_csv(GT_PATH)

agreement_map = {"agree":0, "partially agree":1, "disagree":2}
severity_map  = {"critical":0, "moderate":1, "negligible":2}
edit_map      = {"correction":0, "addition":1, "clarification":2}

gt_df["agreement_label"] = gt_df["agreement"].map(agreement_map)
gt_df["severity_label"]  = gt_df["severity"].map(severity_map)
gt_df["edit_label"]      = gt_df["edit_type"].map(edit_map)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
text_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DebertaV2Model(
  (embeddings): DebertaV2Embeddings(
    (word_embeddings): Embedding(128100, 768, padding_idx=0)
    (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): DebertaV2Encoder(
    (layer): ModuleList(
      (0-11): 12 x DebertaV2Layer(
        (attention): DebertaV2Attention(
          (self): DisentangledSelfAttention(
            (query_proj): Linear(in_features=768, out_features=768, bias=True)
            (key_proj): Linear(in_features=768, out_features=768, bias=True)
            (value_proj): Linear(in_features=768, out_features=768, bias=True)
            (pos_dropout): Dropout(p=0.1, inplace=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): DebertaV2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

In [ ]:
def get_text_embedding(text):
    # Handle empty text safely
    if text is None or len(text.strip()) == 0:
        return np.zeros(768, dtype=np.float32)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = text_model(**inputs)
          # ==============================
    # IMPORTANT FIX: MEAN POOLING (NOT CLS)
    # ==============================
    emb = outputs.last_hidden_state.mean(dim=1)

    # ==============================
    # NORMALIZATION (IMPORTANT)
    # ==============================
    emb = F.normalize(emb, p=2, dim=1)

    return emb.cpu().numpy().squeeze().astype(np.float32)

In [ ]:
def load_ct_series(case_id, series_names=["ABD_PEL"],#, "COR_BODY", "SAG_BODY", "SCOUT"],
                   max_slices=50, img_size=128):

    for series_name in series_names:

        path = os.path.join(IMAGE_PATH, str(case_id), series_name)
        if not os.path.exists(path):
            continue

        files = [os.path.join(path, f) for f in os.listdir(path)]

        slices = []
        for f in files:
            try:
                ds = pydicom.dcmread(f)
                if hasattr(ds, "pixel_array"):
                    slices.append(ds)
            except:
                continue

        if len(slices) == 0:
            continue

        # If valid series found → use it
        slices.sort(key=lambda x: getattr(x, "InstanceNumber", 0))

        imgs = np.array([
            s.pixel_array.astype(np.float32) *
            getattr(s, "RescaleSlope", 1.0) +
            getattr(s, "RescaleIntercept", 0.0)
            for s in slices
        ])

        imgs = np.clip(imgs, -1000, 1000)
        imgs = (imgs + 1000) / 2000.0

        imgs = np.stack([
            resize(sl, (img_size, img_size), anti_aliasing=True)
            for sl in imgs
        ])

        if imgs.shape[0] >= max_slices:
            idx = np.linspace(0, imgs.shape[0]-1, max_slices).astype(int)
            imgs = imgs[idx]
        else:
            pad = max_slices - imgs.shape[0]
            imgs = np.concatenate([
                imgs,
                np.zeros((pad, img_size, img_size), dtype=np.float32)
            ], axis=0)

        return imgs.astype(np.float32)

    return None

In [ ]:
def load_ct_series(case_id, series_names=["ABD_PEL"],#, "ABD_PEL","COR_BODY", "SAG_BODY", "SCOUT"],
                   max_slices=50, img_size=128):

    for series_name in series_names:

        path = os.path.join(IMAGE_PATH, str(case_id), series_name)
        if not os.path.exists(path):
            continue

        files = [os.path.join(path, f) for f in os.listdir(path)]

        slices = []
        for f in files:
            try:
                ds = pydicom.dcmread(f)
                if hasattr(ds, "pixel_array"):
                    slices.append(ds)
            except:
                continue

        if len(slices) == 0:
            continue

        # If valid series found → use it
        slices.sort(key=lambda x: getattr(x, "InstanceNumber", 0))

        imgs = np.array([
            s.pixel_array.astype(np.float32) *
            getattr(s, "RescaleSlope", 1.0) +
            getattr(s, "RescaleIntercept", 0.0)
            for s in slices
        ])

        imgs = np.clip(imgs, -1000, 1000)
        imgs = (imgs + 1000) / 2000.0

        imgs = np.stack([
            resize(sl, (img_size, img_size), anti_aliasing=True)
            for sl in imgs
        ])

        if imgs.shape[0] >= max_slices:
            idx = np.linspace(0, imgs.shape[0]-1, max_slices).astype(int)
            imgs = imgs[idx]
        else:
            pad = max_slices - imgs.shape[0]
            imgs = np.concatenate([
                imgs,
                np.zeros((pad, img_size, img_size), dtype=np.float32)
            ], axis=0)

        return imgs.astype(np.float32)

    return None

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom
from transformers import AutoTokenizer, AutoModel
from skimage.transform import resize

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv3d(1, 8, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(8, 16, 3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1)
        )

        self.fc = nn.Sequential(
            nn.Linear(16, 512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        # IMPORTANT: normalize for fusion stability
        x = F.normalize(x, p=2, dim=1)

        return x


cnn_model = SimpleCNN().to(device)
cnn_model.eval()

SimpleCNN(
  (conv): Sequential(
    (0): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): ReLU()
    (2): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (4): ReLU()
    (5): AdaptiveAvgPool3d(output_size=1)
  )
  (fc): Sequential(
    (0): Linear(in_features=16, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
)

In [ ]:
def get_text_embedding(text):

    # ==============================
    # HANDLE EMPTY TEXT
    # ==============================
    if text is None or len(text.strip()) == 0:
        return np.zeros(768, dtype=np.float32)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = text_model(**inputs)

    # ==============================
    # FIX: MEAN POOLING (IMPORTANT)
    # ==============================
    emb = outputs.last_hidden_state.mean(dim=1)

    # ==============================
    # NORMALIZATION (STABILITY)
    # ==============================
    emb = F.normalize(emb, p=2, dim=1)

    return emb.cpu().numpy().squeeze().astype(np.float32)

In [ ]:
train_data = []

for case in edits_data:
    case_id = case["case_id"]

    # ==============================
    # REPORT
    # ==============================
    try:
        report_text = open(f"{REPORT_PATH}/{case_id}.txt").read()
    except:
        report_text = ""

    report_emb = get_text_embedding(report_text)

    # ==============================
    # CT FEATURES
    # ==============================
    ct_volume = load_ct_series(case_id)

    if ct_volume is not None:
        ct_tensor = torch.tensor(ct_volume).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            ct_emb = cnn_model(ct_tensor).cpu().numpy().squeeze().astype(np.float32)
    else:
        ct_emb = np.zeros(512, dtype=np.float32)

    # IMPORTANT: normalize CT once
    ct_emb = ct_emb / (np.linalg.norm(ct_emb) + 1e-6)

    # ==============================
    # EDIT LOOP
    # ==============================
    for edit in case["edits"]:
        edit_id = edit["edit_id"]

        row = gt_df[gt_df["edit_id"] == edit_id]
        if len(row) == 0:
            continue
        row = row.iloc[0]

        edit_emb = get_text_embedding(edit["suggested_edit_text"])

        # ==============================
        # 🔥 CORE IMPROVEMENT: INTERACTION FEATURES
        # ==============================
        diff = report_emb - edit_emb
        prod = report_emb * edit_emb

        text = np.concatenate([
            report_emb,
            edit_emb,
            diff,
            prod
        ])

        # ==============================
        # FINAL FUSION (REDUCED CT IMPACT)
        # ==============================
        x = np.concatenate([
            text,
            0.3 * ct_emb
        ])

        train_data.append({
            "x": x.astype(np.float32),
            "y": (
                int(row["agreement_label"]),
                int(row["severity_label"]),
                int(row["edit_label"])
            )
        })

In [ ]:
class RadarDataset(torch.utils.data.Dataset):
    def __init__(self, data):

        self.X = torch.tensor(
            [d["x"] for d in data],
            dtype=torch.float32
        )

        self.y1 = torch.tensor(
            [d["y"][0] for d in data],
            dtype=torch.long
        )

        self.y2 = torch.tensor(
            [d["y"][1] for d in data],
            dtype=torch.long
        )

        self.y3 = torch.tensor(
            [d["y"][2] for d in data],
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y1[i], self.y2[i], self.y3[i]


loader = torch.utils.data.DataLoader(
    RadarDataset(train_data),
    batch_size=8,
    shuffle=True
)

/tmp/ipykernel_2948/2628072985.py:4: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.X = torch.tensor(


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionModel(nn.Module):
    def __init__(self, dim):
        super().__init__()

        # ==============================
        # INPUT NORMALIZATION (IMPORTANT)
        # ==============================
        self.norm = nn.LayerNorm(dim)

        # ==============================
        # SHARED FEATURE TRUNK
        # ==============================
        self.shared = nn.Sequential(
            nn.Linear(dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # ==============================
        # TASK-SPECIFIC HEADS
        # ==============================
        self.a = nn.Linear(512, 3)
        self.s = nn.Linear(512, 3)
        self.e = nn.Linear(512, 3)

    def forward(self, x):

        # normalize input
        x = self.norm(x)

        # shared representation
        x = self.shared(x)

        # task outputs
        return self.a(x), self.s(x), self.e(x)


model = FusionModel(train_data[0]["x"].shape[0]).to(device)

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):

    model.train()

    total_loss = 0

    for X, y1, y2, y3 in loader:
        X = X.to(device)
        y1 = y1.to(device)
        y2 = y2.to(device)
        y3 = y3.to(device)

        opt.zero_grad()

        a, s, e = model(X)

        loss_a = loss_fn(a, y1)
        loss_s = loss_fn(s, y2)
        loss_e = loss_fn(e, y3)

        loss = (
            1.5 * loss_a +
            1.2 * loss_s +
            1.0 * loss_e
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        opt.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {total_loss:.4f} | "
        f"A:{loss_a.item():.3f} S:{loss_s.item():.3f} E:{loss_e.item():.3f}"
    )

Epoch 1 | Loss: 83.7771 | A:0.712 S:0.825 E:0.616
Epoch 2 | Loss: 79.4814 | A:0.806 S:1.116 E:0.959
Epoch 3 | Loss: 75.3918 | A:0.675 S:0.890 E:1.074
Epoch 4 | Loss: 73.4813 | A:1.029 S:1.003 E:1.152
Epoch 5 | Loss: 68.9964 | A:0.634 S:0.944 E:0.643


In [ ]:
dim = train_data[0]["x"].shape[0]

class FusionModel(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.shared = nn.Sequential(
            nn.Linear(dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU()
        )

        self.a = nn.Linear(512, 3)
        self.s = nn.Linear(512, 3)
        self.e = nn.Linear(512, 3)

    def forward(self, x):
        x = self.norm(x)
        x = self.shared(x)
        return self.a(x), self.s(x), self.e(x)

model = FusionModel(dim).to(device)

In [ ]:
TEXT_DIM = report_emb.shape[0]   # 768 (DeBERTa)
CT_DIM = 512

INPUT_DIM = TEXT_DIM * 2 + CT_DIM   # 2048
print(INPUT_DIM)

2048


In [ ]:
class FusionModel(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.shared = nn.Sequential(
            nn.Linear(dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU()
        )

        self.a = nn.Linear(512, 3)
        self.s = nn.Linear(512, 3)
        self.e = nn.Linear(512, 3)

    def forward(self, x):
        x = self.norm(x)
        x = self.shared(x)
        return self.a(x), self.s(x), self.e(x)

In [ ]:
model = FusionModel(INPUT_DIM).to(device)

In [ ]:
model.eval()
results = []

with torch.no_grad():

    for case in edits_data:
        case_id = case["case_id"]

        # ==============================
        # REPORT EMBEDDING
        # ==============================
        try:
            report_text = open(f"{REPORT_PATH}/{case_id}.txt").read()
        except:
            report_text = ""

        report_emb = get_text_embedding(report_text)
        report_emb = report_emb / (np.linalg.norm(report_emb) + 1e-6)

        # ==============================
        # CT EMBEDDING
        # ==============================
        ct_volume = load_ct_series(case_id)

        if ct_volume is not None:
            ct_tensor = torch.tensor(ct_volume, dtype=torch.float32)\
                            .unsqueeze(0).unsqueeze(0).to(device)

            ct_emb = cnn_model(ct_tensor).cpu().numpy().squeeze().astype(np.float32)
        else:
            ct_emb = np.zeros(512, dtype=np.float32)

        ct_emb = ct_emb / (np.linalg.norm(ct_emb) + 1e-6)

        # ==============================
        # EDIT LOOP
        # ==============================
        for edit in case["edits"]:
            edit_text = edit.get("suggested_edit_text", "")

            edit_emb = get_text_embedding(edit_text)
            edit_emb = edit_emb / (np.linalg.norm(edit_emb) + 1e-6)

            # ==============================
            # FEATURE VECTOR (SAFE + CONSISTENT)
            # ==============================
            x = np.concatenate([
                report_emb.astype(np.float32),
                edit_emb.astype(np.float32),
                ct_emb.astype(np.float32)
            ]).astype(np.float32)

            # IMPORTANT: enforce correct shape consistency
            X = torch.from_numpy(x).float().unsqueeze(0).to(device)

            # ==============================
            # MODEL PREDICTION
            # ==============================
            a, s, e = model(X)

            results.append({
                "edit_id": edit["edit_id"],
                "agreement": int(a.argmax(dim=1).item()),
                "severity": int(s.argmax(dim=1).item()),
                "edit_type": int(e.argmax(dim=1).item())
            })

In [ ]:
df = pd.DataFrame(results)

In [ ]:
df = df.rename(columns={
    "agreement": "agreement_pred",
    "severity": "severity_pred",
    "edit_type": "edit_pred"
})

In [ ]:
eval_df = gt_df.merge(df, on="edit_id", how="inner")

In [ ]:
agreement_acc = (eval_df["agreement_label"] == eval_df["agreement_pred"]).mean()
severity_acc  = (eval_df["severity_label"]  == eval_df["severity_pred"]).mean()
edit_acc      = (eval_df["edit_label"]      == eval_df["edit_pred"]).mean()

In [ ]:
print("\n=== FINAL RESULTS ===")
print(f"Agreement : {agreement_acc:.4f}")
print(f"Severity  : {severity_acc:.4f}")
print(f"EditType  : {edit_acc:.4f}")
print(f"Average   : {(agreement_acc+severity_acc+edit_acc)/3:.4f}")


=== FINAL RESULTS ===
Agreement : 0.0598
Severity  : 0.4348
EditType  : 0.1739
Average   : 0.2228


In [ ]:
print("\n=== FINAL RESULTS ===")
print(f"Agreement : {agreement_acc:.4f}")
print(f"Severity  : {severity_acc:.4f}")
print(f"EditType  : {edit_acc:.4f}")
print(f"Average   : {(agreement_acc+severity_acc+edit_acc)/3:.4f}")


=== FINAL RESULTS ===
Agreement : 0.2826
Severity  : 0.2500
EditType  : 0.5870
Average   : 0.3732


In [ ]:
print("\n=== FINAL RESULTS ===")
print(f"Agreement : {agreement_acc:.4f}")
print(f"Severity  : {severity_acc:.4f}")
print(f"EditType  : {edit_acc:.4f}")
print(f"Average   : {(agreement_acc+severity_acc+edit_acc)/3:.4f}")


=== FINAL RESULTS ===
Agreement : 0.0598
Severity  : 0.2717
EditType  : 0.2554
Average   : 0.1957


In [ ]:
print("\n=== FINAL RESULTS ===")
print(f"Agreement : {agreement_acc:.4f}")
print(f"Severity  : {severity_acc:.4f}")
print(f"EditType  : {edit_acc:.4f}")
print(f"Average   : {(agreement_acc+severity_acc+edit_acc)/3:.4f}")


=== FINAL RESULTS ===
Agreement : 0.4076
Severity  : 0.5109
EditType  : 0.5815
Average   : 0.5000


In [ ]:
print("\n=== FINAL RESULTS ===")
print(f"Agreement : {agreement_acc:.4f}")
print(f"Severity  : {severity_acc:.4f}")
print(f"EditType  : {edit_acc:.4f}")
print(f"Average   : {(agreement_acc+severity_acc+edit_acc)/3:.4f}")


=== FINAL RESULTS ===
Agreement : 0.4185
Severity  : 0.2826
EditType  : 0.2554
Average   : 0.3188


In [ ]:
df = pd.DataFrame(results)
df.to_csv("/content/drive/My Drive/RADAR-main/RADAR-main/traincnnsubmission.csv", index=False)

# Merge with GT
eval_df = gt_df.merge(df, on="edit_id")

def acc(col):
    return (eval_df[col] == eval_df[col+"_pred"]).mean()

print("\n=== FINAL RESULTS ===")
print("Agreement:", acc("agreement"))
print("Severity :", acc("severity"))
print("EditType :", acc("edit_type"))


=== FINAL RESULTS ===
Agreement: 0.7445652173913043
Severity : 0.5543478260869565
EditType : 0.6032608695652174


In [ ]:
##ABOVE RESULT ACHIEVE ON VALID DATA RECHECK
df = pd.DataFrame(results)
df.to_csv("/content/drive/My Drive/RADAR-main/RADAR-main/traincnnsubmission.csv", index=False)

# Merge with GT
eval_df = gt_df.merge(df, on="edit_id")

def acc(col):
    return (eval_df[col] == eval_df[col+"_pred"]).mean()

print("\n=== FINAL RESULTS ===")
print("Agreement:", acc("agreement"))
print("Severity :", acc("severity"))
print("EditType :", acc("edit_type"))


=== FINAL RESULTS ===
Agreement: 0.7228260869565217
Severity : 0.5543478260869565
EditType : 0.6141304347826086


In [ ]:
# ==============================
# PATHS (UPDATE IF NEEDED)
# ==============================
TEST_BASE_PATH = "/content/drive/My Drive/MediqaRadar02Test"
TEST_IMAGE_PATH = os.path.join(TEST_BASE_PATH, "image_test")
TEST_REPORT_PATH = os.path.join(TEST_BASE_PATH, "test_preliminary_report")
TEST_EDITS_JSON = os.path.join(TEST_BASE_PATH, "test_edits.json")

In [ ]:
# ==============================
# LOAD EDITS
# ==============================
with open(TEST_EDITS_JSON, "r") as f:
    TEST_edits_data = json.load(f)

print(f"Loaded {len(TEST_edits_data)} cases")


Loaded 29 cases


In [ ]:
import json

# print dataset structure
print("Total cases:", len(TEST_edits_data))

# show first case
print("\n=== FIRST CASE ===")
print(TEST_edits_data[0])

# show edits inside it
print("\n=== EDITS OF FIRST CASE ===")
for i, edit in enumerate(TEST_edits_data[0]["edits"][:3]):
    print(f"\nEdit {i+1}:")
    print(edit)

Total cases: 29

=== FIRST CASE ===
{'case_id': 10001, 'clinical_indication': 'small bowel mass seen on push, eval for neoplasm, bleeding', 'preliminary_report_time': 'daytime', 'edits': [{'edit_id': '10001_01', 'suggested_edit_text': 'A few prominent mediastinal lymph nodes without adenopathy.'}, {'edit_id': '10001_02', 'suggested_edit_text': 'The prostate is markedly enlarged with a suspicious soft tissue mass replacing normal architecture.'}, {'edit_id': '10001_03', 'suggested_edit_text': 'There is a moderate right-sided pleural effusion.'}, {'edit_id': '10001_04', 'suggested_edit_text': 'Sclerotic foci in the right iliac bone and right sacral bone, likely representing bone islands, and mild degenerative changes of the spine are present.'}, {'edit_id': '10001_05', 'suggested_edit_text': 'Calcified granulomas in various locations in the lungs and a noncalcified nodule in the right upper lobe.'}, {'edit_id': '10001_06', 'suggested_edit_text': 'Minimal mesenteric stranding associated w

In [ ]:
sample = TEST_edits_data[0]

print("Keys in case:", sample.keys())
print("\nEdit keys:", sample["edits"][0].keys())

Keys in case: dict_keys(['case_id', 'clinical_indication', 'preliminary_report_time', 'edits'])

Edit keys: dict_keys(['edit_id', 'suggested_edit_text'])


In [ ]:
agreement_map = {
    0: "agree",
    1: "partially agree",
    2: "disagree"
}

severity_map = {
    0: "critical",
    1: "moderate",
    2: "negligible"
}

edit_map = {
    0: "correction",
    1: "addition",
    2: "clarification"
}

In [ ]:
model.eval()
results = []

for case in TEST_edits_data:
    case_id = case["case_id"]

    # ==============================
    # REPORT
    # ==============================
    try:
        report_text = open(f"{TEST_REPORT_PATH}/{case_id}.txt").read()
    except:
        report_text = ""

    report_emb = get_text_embedding(report_text)
    report_emb = report_emb / (np.linalg.norm(report_emb) + 1e-6)

    # ==============================
    # CT
    # ==============================
    ct_volume = load_ct_series(case_id)

    if ct_volume is not None:
        ct_tensor = torch.tensor(ct_volume, dtype=torch.float32)\
                        .unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            ct_emb = cnn_model(ct_tensor).cpu().numpy().squeeze().astype(np.float32)
    else:
        ct_emb = np.zeros(512, dtype=np.float32)

    ct_emb = ct_emb / (np.linalg.norm(ct_emb) + 1e-6)

    # ==============================
    # EDIT LOOP
    # ==============================
    for edit in case["edits"]:
        edit_text = edit.get("suggested_edit_text", "")

        edit_emb = get_text_embedding(edit_text)
        edit_emb = edit_emb / (np.linalg.norm(edit_emb) + 1e-6)

        # ==============================
        # FEATURE VECTOR
        # ==============================
        x = np.concatenate([report_emb, edit_emb, ct_emb]).astype(np.float32)
        X = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

        # ==============================
        # PREDICTION
        # ==============================
        with torch.no_grad():
            a, s, e = model(X)

        results.append({
            "edit_id": edit["edit_id"],
            "agreement": int(a.argmax(dim=1).item()),
            "severity": int(s.argmax(dim=1).item()),
            "edit_type": int(e.argmax(dim=1).item())
        })

In [ ]:
df = pd.DataFrame(results)
df.to_csv("/content/drive/My Drive/MediqaRadar02Test/testDeberta04SAGsubmission.csv", index=False)

print("Saved submission.csv")
print(df.head())

Saved submission.csv
    edit_id  agreement  severity  edit_type
0  10001_01          0         0          1
1  10001_02          0         0          1
2  10001_03          2         0          1
3  10001_04          0         0          1
4  10001_05          0         0          1


In [ ]:
results

[{'edit_id': '10001_01', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_02', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_03', 'agreement': 2, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_04', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_05', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_06', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_07', 'agreement': 2, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_08', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_09', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_10', 'agreement': 1, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_11', 'agreement': 0, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10001_12', 'agreement': 2, 'severity': 0, 'edit_type': 1},
 {'edit_id': '10002_01', 'agreement': 1, 'severity': 1, 'edit_type': 1},
 {'edit_id': '10002_02', 'agreement': 2, 'severity'

In [ ]:
import pandas as pd

# load your saved file
df = pd.read_csv("/content/drive/MyDrive/MediqaRadar02Test/testDeberta04SAGsubmission.csv")

# label maps (IMPORTANT: must match training)
agreement_map = {
    0: "agree",
    1: "partially agree",
    2: "disagree"
}

severity_map = {
    0: "critical",
    1: "moderate",
    2: "negligible"
}

edit_map = {
    0: "correction",
    1: "addition",
    2: "clarification"
}

# convert back to strings
df["agreement_pred"] = df["agreement"].map(agreement_map)
df["severity_pred"] = df["severity"].map(severity_map)
df["edit_type_pred"] = df["edit_type"].map(edit_map)

# save final submission file
df.to_csv("/content/drive/MyDrive/MediqaRadar02Test/testDeberta04SAGsubmission_final.csv", index=False)

print(df.head())

    edit_id  agreement  severity  edit_type agreement_pred severity_pred  \
0  10001_01          0         0          1          agree      critical   
1  10001_02          0         0          1          agree      critical   
2  10001_03          2         0          1       disagree      critical   
3  10001_04          0         0          1          agree      critical   
4  10001_05          0         0          1          agree      critical   

  edit_type_pred  
0       addition  
1       addition  
2       addition  
3       addition  
4       addition  
